# Домашнее задание 1. Autoencoders & Fréchet Inception Distance

В этом задании вы:
1. Напишете свой автоэнкодер (AE) для датасета CIFAR10.
2. Используете эмбеддинги энкодера, чтобы посчитать Fréchet Inception Distance (FID) между классами CIFAR10.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Subset, DataLoader
from torchvision import datasets, transforms

from sklearn.ensemble import GradientBoostingClassifier

from tqdm.auto import tqdm, trange

from collections import defaultdict
import random

In [2]:
def set_seed(seed: int):
    """
    Sets the seed for reproducibility across numpy, random, torch.

    Parameters:
    seed (int): The seed value to be set.
    """
    random.seed(seed)  # Python's random module
    np.random.seed(seed)  # NumPy
    torch.manual_seed(seed)  # Torch for CPU and device-agnostic
    torch.cuda.manual_seed(seed)  # GPU: CUDA 
    torch.cuda.manual_seed_all(seed)  # Multi-GPU
    torch.backends.cudnn.deterministic = True  # Deterministic behavior for CUDA
    torch.backends.cudnn.benchmark = False  # Disable cuDNN benchmarking

SEED = 42
set_seed(SEED)

In [ ]:
device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
device

Для работы с данными будем использовать `torchvision`.

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: (x * 2) - 1
])

In [ ]:
train_dataset = datasets.CIFAR10(
    '../data/cifar',
    train=True,
    transform=transform,
    download=True)
val_dataset = datasets.CIFAR10(
    '../data/cifar',
    train=False,
    transform=transform,
    download=True)
len(train_dataset), len(val_dataset)

Нам нужно будет уметь «разнормализовать» картинки (обратно в $[0,1]$) для визуализации.

In [6]:
def denormalize_image(image):
    return (image + 1) / 2

In [ ]:
text_labels = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
plt.figure(figsize=(10, 10))
for index, (image, label) in enumerate(train_dataset):
    plt.subplot(5, 5, index + 1)
    plt.imshow(denormalize_image(image.permute(1, 2, 0)))
    plt.axis('off')
    plt.title(text_labels[label])
    if index == 24: break
plt.show()

Размерность картинки в CIFAR10: 3 канала, 32×32 пикселя.

In [ ]:
image.shape

In [ ]:
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
len(train_loader), len(val_loader)

## Задание 1. Обучить AE (3 балла)

Постройте свой AE: напишите классы `Encoder` и `Decoder`. Можно использовать любые слои, которые посчитаете нужными: `nn.Conv2d`, `nn.AvgPool2d`, `nn.MaxPool2d`, а также в декодере — `nn.Upsample`, `nn.ConvTranspose2d` и т.д.  

![](https://miro.medium.com/max/1400/1*44eDEuZBEsmG_TCAKRI3Kw@2x.png)

In [11]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        ### BEGIN SOLUTION
        # Ваш код здесь
        ### END SOLUTION

    def forward(self, x):
        ### BEGIN SOLUTION
        # Ваш код здесь
        ### END SOLUTION

In [12]:
encoder = Encoder()
noise = torch.rand(1, 3, 32, 32) - 1
assert encoder(noise).view(-1).shape[0] < 1*3*32*32, "Размер эмбеддинга должен быть меньше исходного!"

In [13]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        ### BEGIN SOLUTION
        # Ваш код здесь
        ### END SOLUTION

    def forward(self, z):
        ### BEGIN SOLUTION
        # Ваш код здесь
        ### END SOLUTION

In [14]:
decoder = Decoder()
noise = torch.rand(1, 3, 32, 32)
emb = encoder(noise)
assert decoder(emb).shape == (1, 3, 32, 32), "Декодер должен восстанавливать тензор формы (batch_size, 3, 32, 32)"

Для оценки качества энкодера возьмём его выходы как фичи для классификации (Gradient Boosting). Затем сравним скор классификации для необученного энкодера со скором обученного. Для ускорения расчета, мы используем только часть трейна.

In [15]:
def classification_score(encoder, _train_dataset, val_dataset, device, train_size=5000):
    encoder.eval()
    
    # Возьмём train_size случайных примеров для обучения GBDT
    train_dataset = Subset(_train_dataset, torch.randperm(len(_train_dataset))[:train_size])
    
    X_train = []
    y_train = []
    for image, label in tqdm(train_dataset):
        image = image.to(device)
        with torch.no_grad():
            emb = encoder(image[None, ...])
        X_train.append(emb.cpu().numpy().reshape(-1))
        y_train.append(label)
    X_train = np.stack(X_train)
    y_train = np.stack(y_train)
    
    clf = GradientBoostingClassifier(n_estimators=50, max_depth=5, verbose=1, random_state=SEED)
    clf.fit(X_train, y_train)
    
    X_val = []
    y_val = []
    for image, label in tqdm(val_dataset):
        image = image.to(device)
        with torch.no_grad():
            emb = encoder(image[None, ...])
        X_val.append(emb.cpu().numpy().reshape(-1))
        y_val.append(label)
    X_val = np.stack(X_val)
    y_val = np.stack(y_val)
    
    return clf.score(X_val, y_val)

In [ ]:
pretrain_score = classification_score(Encoder().to(device), train_dataset, val_dataset, device)
pretrain_score

Напишите функцию `train`, которая обучает энкодер и декодер на **всём трейне**, возвращая средний MSE.

In [19]:
def train(loader, optim, encoder, decoder, device):
    encoder.train()
    decoder.train()
    criterion = nn.MSELoss()
    losses = []
    for image, _ in tqdm(loader, leave=False):
        ### BEGIN SOLUTION
        # Ваш код здесь
        ### END SOLUTION
    return sum(losses) / len(losses)

In [ ]:
encoder.to(device)
decoder.to(device)

params = list(encoder.parameters()) + list(decoder.parameters())
optim = torch.optim.AdamW(params)

loss = train(train_loader, optim, encoder, decoder, device)
assert type(loss) == float, "Функция train должна возвращать float"
assert 0 < loss < 1
loss

Напишите функцию `eval`, которая возвращает среднее MSE на всём валидационном датасете.  
Hint: Не забудьте выключить расчёт градиентов.

In [20]:
def eval(loader, encoder, decoder, device):
    encoder.eval()
    decoder.eval()
    criterion = nn.MSELoss()
    losses = []
    with torch.no_grad():
        for image, _ in tqdm(loader, leave=False):
            ### BEGIN SOLUTION
            # Ваш код здесь
            ### END SOLUTION
    return sum(losses) / len(losses)

In [ ]:
loss = eval(val_loader, encoder, decoder, device)
assert type(loss) == float
assert 0 < loss < 1
loss

Ниже — функция `full_train`, которая возвращает обученныe энкодер и декодер. Обучите модель, а затем добавьте загрузку предобученных весов в начало функции.

In [48]:
def full_train(device, train_loader, val_loader):
    ### BEGIN SOLUTION
    # Ваш код здесь
    ### END SOLUTION
    
    encoder = Encoder().to(device)
    decoder = Decoder().to(device)
    
    params = list(encoder.parameters()) + list(decoder.parameters())
    optim = torch.optim.AdamW(params)
    train_loss = []
    val_loss = []
    n_epochs = 30
    for e in trange(n_epochs):
        e_train_loss = train(train_loader, optim, encoder, decoder, device)
        train_loss.append(e_train_loss)
        e_val_loss = eval(val_loader, encoder, decoder, device)
        val_loss.append(e_val_loss)
        print(f'Epoch: {e+1}/{n_epochs}')
        print(f'Train MSE loss: {e_train_loss:.4f}')
        print(f'Validation MSE loss: {e_val_loss:.4f}')
    plt.plot(train_loss, label='train')
    plt.plot(val_loss, label='val')
    plt.legend()
    plt.title('MSE Loss')
    plt.grid()
    plt.show()
    return encoder, decoder

In [ ]:
encoder, decoder = full_train(device, train_loader, val_loader)

In [ ]:
score = classification_score(encoder, train_dataset, val_dataset, device, train_size=5000)
assert score > pretrain_score * 1.05, "Слишком низкий скор. Убедитесь, что ваш AE обучается."
score

In [ ]:
encoder.eval()
decoder.eval()
plt.figure(figsize=(5, 25))
for index, (image, label) in enumerate(val_loader):
    plt.subplot(10, 2, index*2 + 1)
    plt.imshow(denormalize_image(image)[0].permute(1, 2, 0))
    plt.axis('off')
    plt.title(text_labels[label])
    
    plt.subplot(10, 2, index*2 + 2)
    image = image.to(device)
    with torch.no_grad():
        emb = encoder(image)
        rec = decoder(emb).cpu()
    plt.imshow(denormalize_image(rec)[0].permute(1, 2, 0))
    plt.axis('off')
    
    if index == 9:
        break

## Задание 2. FID дистанция между классами CIFAR10 (3 балла)

Теперь, используя bottleneck-представления обученного энкодера, посчитаем **Fréchet Inception Distance (FID)** между разными классами CIFAR10 на **валидационной** выборке.

Напомним формулу FID:
$$
\mathrm{FID} = \|\mu_r - \mu_g\|^2 + \operatorname{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r \Sigma_g)^{1/2}).
$$

Реализуйте функцию `get_representations`, которая для заданного `DataLoader` и энкодера вернёт `defaultdict`, где:
- ключ: номер класса
- значение: список тензоров, соответствующих эмбеддингам картинок этого класса.

In [38]:
def get_representations(dataloader, encoder, device):
    representations = defaultdict(list)
    ### BEGIN SOLUTION
    # Ваш код здесь
    ### END SOLUTION

In [ ]:
representations = get_representations(val_loader, encoder, device)
assert len(representations) == 10, "Должно быть 10 ключей по количеству классов."
assert len(representations[0]) == 1000, "Количество эмбеддингов на класс не совпадает с ожидаемым."
assert type(representations[0][0]) == torch.Tensor

In [57]:
def calculate_fid(repr1, repr2, eps=1e-6):
    ### BEGIN SOLUTION
    # Ваш код здесь
    ### END SOLUTION

In [ ]:
# Посчитаем FID для каждой пары классов и сохраним в heatmap (10×10)
heatmap = np.zeros((10, 10))
for label_from in trange(10):
    for label_to in range(10):
        fid = calculate_fid(
            torch.stack(representations[label_from], dim=0).cpu().numpy(),
            torch.stack(representations[label_to], dim=0).cpu().numpy()
        )
        heatmap[label_from, label_to] = fid

# Проверки на корректность вычислений:
assert heatmap.shape == (10, 10), "Массив должен быть 10×10."
assert np.all(heatmap + 1e-5 > 0), "FID не может быть отрицательной."
airplane_ship = heatmap[0, 8]  # fid(airplane, ship)
airplane_frog = heatmap[0, 6]  # fid(airplane, frog)
truck_automobile = heatmap[9, 1]  # fid(truck, automobile)
truck_dog = heatmap[9, 5]  # fid(truck, dog)
assert airplane_ship < airplane_frog, "Ожидаем, что класс airplane ближе к ship, чем к frog"
assert truck_automobile < truck_dog, "Ожидаем, что класс truck ближе к automobile, чем к dog"

In [ ]:
sns.heatmap(
    heatmap, 
    linewidth=0.5, 
    xticklabels=text_labels, 
    yticklabels=text_labels
)
plt.title('FID-дистанция между классами на валидации CIFAR10')
plt.show()

## Задание 3 (3 балла)

Возьмите **любой** датасет из интернета, в котором есть **два разных класса** (например, «кошки и собаки», или какие-то другие). Посчитайте между ними FID, используя **тот же энкодер** (обученный на CIFAR10).

In [ ]:
# YOUR CODE HERE

## Задание 4 (1 балл)

Поздравляем, вы проделали большую работу! В этом разделе:
1. Опишите идеи, которые вы хотели бы попробовать, если бы у вас было больше времени или статьи, в которых решают подобные задачи.  
2. Приложите какой-нибудь свежий (последних пары лет) мем про генеративные модели. Если мем будет не смешным, увы, придётся снять баллы.

In [ ]:
# YOUR CODE HERE